# Parameters

In [ ]:
RAW_FOLDER = 'raw_data/'
DEMO_FILE = 'demo_patient_covid.csv'
LAB_FILE = 'fact_lab_covid_first.csv'
CHOSEN_LABS_FILE = '../chosen_lab_exams_list.csv'
OUTPUT_FILE = '../complete_data/patient_covid.csv'

# Import Data

In [ ]:
import pandas as pd
import os

# Set the path to the datasets
current_dir = os.getcwd()
folderpath = os.path.join(current_dir, RAW_FOLDER)

df_lab = pd.read_csv(os.path.join(folderpath, LAB_FILE)).drop(columns=['rn'])
df_demographics = pd.read_csv(os.path.join(folderpath, DEMO_FILE))

## Preprocessing Lab

In [ ]:
df_lab.head()

In [ ]:
df_lab_aux = df_lab[['hadm_id', 'lab_itemid', 'lab_value', 'lab_valuenum']]

# Fill the lab_value column with lab_valuenum asif it's not null, else keep lab_value
df_lab_aux['final_value'] = df_lab_aux['lab_valuenum'].fillna(df_lab_aux['lab_value'])

# Drop unnecessary columns
df_lab_aux = df_lab_aux[['hadm_id', 'lab_itemid', 'final_value']]

# Pivot the table so lab_itemid becomes columns
df_pivot = df_lab_aux.pivot(index='hadm_id', columns='lab_itemid', values='final_value')

# Reset the index to make hadm_id a column again
df_pivot.reset_index(inplace=True)

# Rename columns for clarity (optional)
df_pivot.columns.name = None  # Remove the multi-index name

df_pivot.head()

In [ ]:
# Load lab description data
df_lab_description = pd.read_csv(os.path.join(folderpath, CHOSEN_LABS_FILE))

# Update all labels by appending fluid and category
df_lab_description['label'] = df_lab_description.apply(
    lambda row: f"{row['label']}_{row['category']}_{row['fluid']}_{row['valueuom']}", axis=1
)

# Create a mapping dictionary from itemid to updated label
df_lab_mapping = df_lab_description[['itemid', 'label']]
itemid_to_label = dict(zip(df_lab_mapping['itemid'], df_lab_mapping['label']))

# Rename columns in df_pivot using the updated label mapping
df_pivot.rename(columns=itemid_to_label, inplace=True)

df_pivot.head()

In [ ]:
def convert_column_to_float(df, column_name):
    """Convert a column to float if all non-NaN values conform to float."""
    
    # Check if all non-NaN values can be converted to float
    is_float_convertible = df[column_name].dropna().apply(lambda x: isinstance(x, (int, float)) or str(x).replace('.', '', 1).isdigit()).all()
    
    if is_float_convertible:
        df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    
    return df

In [ ]:
# Convert only columns that are fully numeric (excluding NaNs)
for col in df_pivot.columns:
    df_pivot = convert_column_to_float(df_pivot, col)

# Show DataFrame info with columns sorted in alphabetical order
df_pivot_sorted = df_pivot.reindex(sorted(df_pivot.columns), axis=1)
df_pivot_sorted.info()

### Remove Columns with less than 40% non-null values

In [ ]:
# Identify columns with less than 40% non-null values
columns_to_remove = df_pivot.columns[df_pivot.isnull().sum() > (len(df_pivot) * 0.6)]

# Save these columns separately
df_removed_columns = df_pivot[columns_to_remove]

# Remove these columns from df_pivot
df_pivot = df_pivot.drop(columns=columns_to_remove)

# Display the updated df_pivot and the removed columns
df_removed_columns.info()

In [ ]:
columns_to_remove.tolist()

In [ ]:
# Update the values in 'Protein' and 'Specific Gravity' columns
df_pivot['Specific Gravity_Hematology_Urine_nan'] = df_pivot['Specific Gravity_Hematology_Urine_nan'].apply(lambda x: x if isinstance(x, float) else None)

df_lab_covid = df_pivot.copy()
df_lab_covid.info()

# Demographics

In [ ]:
df_demographics.columns

In [ ]:
a = set(df_demographics["hadm_id"])
b = set(df_lab_covid["hadm_id"])

c = a.intersection(b)
len(c)

# Unify the columns

In [ ]:
df_full_covid = df_demographics.merge(df_lab_covid, on='hadm_id', how='inner')

In [ ]:
df_full_covid['COVID'] = 1
df_full_covid.head()

In [ ]:
df_full_covid.drop(
    columns=[
        "start_year",
        "end_year",
        "race",
        "admission_type",
        "time_died_after",
        "I_Chemistry_Blood_U",
        "L_Chemistry_Blood_U",
        "H_Chemistry_Blood_U",
    ],
    inplace=True,
)

df_full_covid = pd.get_dummies(df_full_covid, columns=["gender"], drop_first=True)

# Convert all boolean columns to integers
bool_columns = df_full_covid.select_dtypes(include=["bool"]).columns
df_full_covid[bool_columns] = df_full_covid[bool_columns].astype(int)

In [ ]:
df_full_covid.to_csv(os.path.join(folderpath, OUTPUT_FILE), index=False)